# Inpaint From Checkpoint

这个 notebook 只保留一条推理路径：

```text
输入 run_config.json + .ckpt -> 固定 reactant(fragment == 0) -> 生成 product(fragment == 1)
```

输出包括：

- `conditioned_reactant.extxyz`
- `predicted_product.extxyz`
- `target_product.extxyz`
- `inpaint_visualization.png`
- `inpaint_summary.json`


## 0. Imports And Local Helper Functions

这里把 checkpoint 重建模型所需的辅助函数全部内联进 notebook，本文件不再依赖 `use_checkpoint_example.py`。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from ase import Atoms
from ase.data import chemical_symbols
from ase.io import write

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "inpaint_from_checkpoint.ipynb").exists():
    NOTEBOOK_DIR = Path("tests/react_product_xyz_training").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
PACKAGE_PARENT = REPO_ROOT.parent
if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

from akmcgc.model import EGNN, LEFTNet
from akmcgc.trainer.task import DiffModule


def build_module_from_config(config: Dict[str, Any]) -> DiffModule:
    split_paths = {key: Path(value) for key, value in config["split_paths"].items()}
    model_type = config["model_type"]
    hidden_nf = int(config["hidden_nf"])
    n_layers = int(config["n_layers"])
    cutoff = float(config["cutoff"])
    node_nf = 3 + 118 + 1
    in_hidden_with_time = hidden_nf + 1

    if model_type == "egnn":
        model = EGNN
        model_config = {
            "in_node_nf": in_hidden_with_time,
            "in_edge_nf": 0,
            "hidden_nf": hidden_nf,
            "edge_hidden_nf": max(16, hidden_nf // 2),
            "act_fn": "swish",
            "n_layers": n_layers,
            "attention": False,
            "out_node_nf": None,
            "tanh": True,
            "coords_range": 5.0,
            "norm_constant": 1.0,
            "inv_sublayers": 1,
            "sin_embedding": False,
            "normalization_factor": 1.0,
            "aggregation_method": "mean",
            "reflect_equiv": True,
        }
    elif model_type == "leftnet":
        model = LEFTNet
        model_config = {
            "pos_require_grad": False,
            "cutoff": cutoff,
            "num_layers": n_layers,
            "hidden_channels": hidden_nf,
            "num_radial": 32,
            "in_hidden_channels": in_hidden_with_time,
            "reflect_equiv": True,
            "legacy": True,
            "update": True,
            "pos_grad": False,
            "single_layer_output": True,
            "object_aware": True,
        }
    else:
        raise ValueError(f"Unknown model_type in config: {model_type}")

    training_config = {
        "train_react_file": str(split_paths["train_react"]),
        "train_product_file": str(split_paths["train_product"]),
        "val_react_file": str(split_paths["val_react"]),
        "val_product_file": str(split_paths["val_product"]),
        "test_react_file": str(split_paths["test_react"]),
        "test_product_file": str(split_paths["test_product"]),
        "cutoff": cutoff,
        "max_neigh": int(config["max_neigh"]),
        "r_fixed": True,
        "r_pbc": True,
        "device": "cpu",
        "dtype": config["dtype"],
        "num_elements": 118,
        "bz": int(config["batch_size"]),
        "num_workers": int(config["num_workers"]),
        "clip_grad": False,
        "lr_schedule_type": None,
        "sampling_timesteps": int(config["sampling_timesteps"]),
    }

    return DiffModule(
        model_config=model_config,
        optimizer_config={"lr": float(config["lr"]), "betas": [0.9, 0.999], "weight_decay": 0.0},
        training_config=training_config,
        node_nfs=[node_nf],
        edge_nf=0,
        condition_nf=int(config.get("condition_nf", 0)),
        fragment_names=["IS", "FS"],
        pos_dim=3,
        update_pocket_coords=True,
        condition_time=True,
        edge_cutoff=None,
        norm_values=(1.0, 1.0, 1.0),
        norm_biases=(0.0, 0.0, 0.0),
        noise_schedule="cosine",
        timesteps=int(config["timesteps"]),
        precision=1e-5,
        loss_type="l2",
        pos_only=bool(config["pos_only"]),
        model=model,
        eval_epochs=10_000,
    )


def atom_symbols_from_batch(batch: Dict[str, torch.Tensor], atom_mask: torch.Tensor) -> list[str]:
    one_hot = batch["h"][atom_mask, 3:121]
    atomic_numbers = torch.argmax(one_hot, dim=1).detach().cpu().tolist()
    return [chemical_symbols[int(index) + 1] for index in atomic_numbers]


def write_extxyz(
    path: Path,
    symbols: list[str],
    positions: torch.Tensor,
    cell: torch.Tensor,
    pbc: torch.Tensor,
    comment: str,
) -> None:
    atoms = Atoms(
        symbols=symbols,
        positions=positions.detach().cpu().numpy(),
        cell=cell.detach().cpu().numpy(),
        pbc=pbc.detach().cpu().numpy().astype(bool),
    )
    atoms.info["comment"] = comment
    write(path, atoms, format="extxyz")


def default_conditions(batch: Dict[str, torch.Tensor], condition_nf: int) -> torch.Tensor | None:
    if condition_nf <= 0:
        return None
    num_graphs = int(batch["mask"].max().item()) + 1
    return torch.zeros((num_graphs, condition_nf), dtype=batch["h"].dtype, device=batch["h"].device)


def sample_mask(batch: Dict[str, torch.Tensor], sample_index: int) -> torch.Tensor:
    sample_ids = torch.unique(batch["mask"], sorted=True)
    if sample_index < 0 or sample_index >= len(sample_ids):
        raise IndexError(f"sample_index={sample_index} out of range for batch with {len(sample_ids)} samples")
    return batch["mask"] == int(sample_ids[sample_index].item())


def rmsd(pred: torch.Tensor, target: torch.Tensor) -> float:
    value = torch.sqrt(torch.mean(torch.sum((pred - target) ** 2, dim=1)))
    return float(value.detach().cpu())


print("notebook dir:", NOTEBOOK_DIR)
print("repo root:   ", REPO_ROOT)

notebook dir: /Users/wx/Desktop/yyxwjq/akmcgc/tests/react_product_xyz_training
repo root:    /Users/wx/Desktop/yyxwjq/akmcgc


## 1. Paths And Runtime Options

默认读取当前测试目录里已经训练好的 `run_config.json` 和 `last.ckpt`。如果你后面训练了新的 run，只改这两个路径即可。

In [2]:
RUN_DIR = NOTEBOOK_DIR / "runs" / "pdau_test"
CONFIG_PATH = RUN_DIR / "run_config.json"
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "last.ckpt"
OUTPUT_DIR = RUN_DIR / "notebook_inpaint"

SAMPLE_INDEX = 0
SEED = 123
INPAINT_TIMESTEPS = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("config:    ", CONFIG_PATH)
print("checkpoint:", CHECKPOINT_PATH)
print("output dir:", OUTPUT_DIR)

config:     /Users/wx/Desktop/yyxwjq/akmcgc/tests/react_product_xyz_training/runs/pdau_test/run_config.json
checkpoint: /Users/wx/Desktop/yyxwjq/akmcgc/tests/react_product_xyz_training/runs/pdau_test/checkpoints/last.ckpt
output dir: /Users/wx/Desktop/yyxwjq/akmcgc/tests/react_product_xyz_training/runs/pdau_test/notebook_inpaint


## 2. Load Config, Module, And One Test Sample

这里会：

1. 读取 `run_config.json`
2. 重新构建 `DiffModule`
3. 加载 `.ckpt`
4. 从 test dataloader 取一个 batch
5. 选出其中第 `SAMPLE_INDEX` 个样本的 reactant / product 节点
6. 同时取出这个样本对应的 `cell / pbc`，用于写出周期体系 `extxyz`

In [3]:
config = json.loads(CONFIG_PATH.read_text())
module = build_module_from_config(config)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
module.load_state_dict(checkpoint["state_dict"])
module.setup("test")
module.eval()

condition_nf = int(config.get("condition_nf", 0))
timesteps = int(config["sampling_timesteps"]) if INPAINT_TIMESTEPS is None else int(INPAINT_TIMESTEPS)

batch = next(iter(module.test_dataloader()))
selected_mask = sample_mask(batch, SAMPLE_INDEX)
reactant_mask = selected_mask & (batch["fragment"] == 0)
product_mask = selected_mask & (batch["fragment"] == 1)

reactant_symbols = atom_symbols_from_batch(batch, reactant_mask)
product_symbols = atom_symbols_from_batch(batch, product_mask)
reactant_cell = batch["cell"][SAMPLE_INDEX, 0]
reactant_pbc = batch["pbc"][SAMPLE_INDEX, 0]
product_cell = batch["cell"][SAMPLE_INDEX, 1]
product_pbc = batch["pbc"][SAMPLE_INDEX, 1]
conditions = default_conditions(batch, condition_nf)

print("run name:     ", config["run_name"])
print("model type:   ", config["model_type"])
print("sample index: ", SAMPLE_INDEX)
print("timesteps:    ", timesteps)
print("reactant atoms:", int(reactant_mask.sum().item()))
print("product atoms: ", int(product_mask.sum().item()))

RuntimeError: Error(s) in loading state_dict for DiffModule:
	Unexpected key(s) in state_dict: "ddpm.denoiser.model.e_block_2.gcl_0.edge_mlp.mlp.0.linear.weight", "ddpm.denoiser.model.e_block_2.gcl_0.edge_mlp.mlp.0.linear.bias", "ddpm.denoiser.model.e_block_2.gcl_0.edge_mlp.mlp.1.linear.weight", "ddpm.denoiser.model.e_block_2.gcl_0.edge_mlp.mlp.1.linear.bias", "ddpm.denoiser.model.e_block_2.gcl_0.node_mlp.mlp.0.linear.weight", "ddpm.denoiser.model.e_block_2.gcl_0.node_mlp.mlp.0.linear.bias", "ddpm.denoiser.model.e_block_2.gcl_0.node_mlp.mlp.1.linear.weight", "ddpm.denoiser.model.e_block_2.gcl_0.node_mlp.mlp.1.linear.bias", "ddpm.denoiser.model.e_block_2.gcl_equiv.distance_embedding.mlp.0.linear.weight", "ddpm.denoiser.model.e_block_2.gcl_equiv.distance_embedding.mlp.0.linear.bias", "ddpm.denoiser.model.e_block_2.gcl_equiv.distance_embedding.mlp.1.linear.weight", "ddpm.denoiser.model.e_block_2.gcl_equiv.distance_embedding.mlp.1.linear.bias", "ddpm.denoiser.model.e_block_2.gcl_equiv.coor

## 3. Run One Inpaint And Export EXTXYZ

固定 `fragment == 0` 的 reactant，只生成 `fragment == 1` 的 product。

因为这里是周期体系，导出时会把 `cell / pbc` 一起写进 `extxyz`。

In [ ]:
torch.manual_seed(SEED)
with torch.no_grad():
    out = module.ddpm.inpaint(
        batch=batch,
        conditions=conditions,
        frag_fixed=[0],
        timesteps=timesteps,
    )

reactant_path = OUTPUT_DIR / "conditioned_reactant.extxyz"
pred_path = OUTPUT_DIR / "predicted_product.extxyz"
target_path = OUTPUT_DIR / "target_product.extxyz"

write_extxyz(
    reactant_path,
    reactant_symbols,
    batch["pos"][reactant_mask],
    reactant_cell,
    reactant_pbc,
    "Condition reactant kept fixed during inpainting",
)
write_extxyz(
    pred_path,
    product_symbols,
    out["pos"][product_mask],
    product_cell,
    product_pbc,
    f"Predicted product via inpaint from checkpoint={CHECKPOINT_PATH.name}",
)
write_extxyz(
    target_path,
    product_symbols,
    batch["pos"][product_mask],
    product_cell,
    product_pbc,
    "Ground-truth product from test split",
)

product_rmsd = rmsd(out["pos"][product_mask], batch["pos"][product_mask])
summary = {
    "config_path": str(CONFIG_PATH),
    "checkpoint_path": str(CHECKPOINT_PATH),
    "output_dir": str(OUTPUT_DIR),
    "sample_index": SAMPLE_INDEX,
    "seed": SEED,
    "timesteps": timesteps,
    "product_rmsd": product_rmsd,
}
(OUTPUT_DIR / "inpaint_summary.json").write_text(json.dumps(summary, indent=2))

print("reactant:", reactant_path)
print("predicted:", pred_path)
print("target:   ", target_path)
print("product RMSD:", product_rmsd)

## 4. Visualize And Export A PNG

这一步会把 reactant、预测 product、真实 product 放到同一张图里，方便你直观看模型输出。

In [ ]:
def to_numpy(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().numpy()


def axis_limits(*arrays: np.ndarray) -> tuple[tuple[float, float], tuple[float, float], tuple[float, float]]:
    merged = np.concatenate(arrays, axis=0)
    mins = merged.min(axis=0)
    maxs = merged.max(axis=0)
    center = (mins + maxs) / 2.0
    radius = float(np.max(maxs - mins) / 2.0)
    radius = max(radius, 1e-6)
    return (
        (center[0] - radius, center[0] + radius),
        (center[1] - radius, center[1] + radius),
        (center[2] - radius, center[2] + radius),
    )


def scatter_atoms(ax, pos: np.ndarray, symbols: list[str], title: str, color: str) -> None:
    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], s=90, c=color, edgecolors="black", linewidths=0.5)
    for xyz, symbol in zip(pos, symbols):
        ax.text(xyz[0], xyz[1], xyz[2], symbol, fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")


reactant_pos = to_numpy(batch["pos"][reactant_mask])
pred_pos = to_numpy(out["pos"][product_mask])
target_pos = to_numpy(batch["pos"][product_mask])
xlim, ylim, zlim = axis_limits(reactant_pos, pred_pos, target_pos)

fig = plt.figure(figsize=(15, 4.8))
axes = [fig.add_subplot(1, 3, i + 1, projection="3d") for i in range(3)]
scatter_atoms(axes[0], reactant_pos, reactant_symbols, "Conditioned Reactant", "tab:blue")
scatter_atoms(axes[1], pred_pos, product_symbols, f"Predicted Product\nRMSD={product_rmsd:.4f}", "tab:orange")
scatter_atoms(axes[2], target_pos, product_symbols, "Target Product", "tab:green")

for ax in axes:
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)

fig.suptitle(f"Inpaint Result From {CHECKPOINT_PATH.name}")
fig.tight_layout()
png_path = OUTPUT_DIR / "inpaint_visualization.png"
fig.savefig(png_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("saved figure:", png_path)